# Notebook 2 – Data Type Handling

Correct data types are the foundation everything else in preprocessing builds on: you can't scale a number stored as text, group by a date stored inconsistently, or feed a raw string into most ML algorithms. This notebook covers how to identify, understand, and convert data types, using the real problems present in `customer_transactions_raw.csv`.

**Note:** the focus here is on *type* correctness. Deeper issues like implausible values (e.g. `rating = -1`) or duplicate rows are addressed in later preprocessing notebooks — here we only fix what's needed to get each column into its correct type.


In [1]:
import pandas as pd
import numpy as np
pd.set_option('display.max_columns', None)
df = pd.read_csv('customer_transactions_raw.csv')
df.head()

,customer_id,age,gender,annual_income,city,membership_type,purchase_amount,quantity,signup_date,payment_method,rating,notes
0,100508,43.0,female,80242.98,Bengaluru,Gold,99.13,1.0,08-22-2021,Debit Card,1,NaN
1,100819,41.0,Female,NaN,Delhi,Silver,113.23,1.0,16 Jun 2020,Net Banking,4,NaN
2,100453,61.0,Male,56285.40,Bengaluru,Gold,248.19,2.0,28 Sep 2020,Debit Card,4,NaN
3,100369,NaN,Male,81878.74,NaN,Silver,51.27,9.0,10 Sep 2023,Credit Card,4,NaN
4,100243,24.0,Male,100566.42,Delhi,Silver,97.97,1.0,07/01/2021,Credit Card,4,NaN


## 1. Identifying Data Types

The first step with any new dataset is to check what type pandas *thinks* each column is — and compare that against what it *should* be conceptually.


In [3]:
df.dtypes

customer_id          int64
age                    str
gender                 str
annual_income      float64
city                   str
membership_type        str
purchase_amount        str
quantity           float64
signup_date            str
payment_method         str
rating               int64
notes              float64
dtype: object

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   customer_id      1000 non-null   int64  
 1   age              956 non-null    str    
 2   gender           910 non-null    str    
 3   annual_income    943 non-null    float64
 4   city             847 non-null    str    
 5   membership_type  937 non-null    str    
 6   purchase_amount  1000 non-null   str    
 7   quantity         980 non-null    float64
 8   signup_date      984 non-null    str    
 9   payment_method   966 non-null    str    
 10  rating           1000 non-null   int64  
 11  notes            0 non-null      float64
dtypes: float64(3), int64(2), str(7)
memory usage: 136.7 KB


Comparing this to what each column conceptually represents already reveals mismatches:

| Column | pandas dtype | Should conceptually be |
|---|---|---|
| `age` | object (string) | Numerical |
| `purchase_amount` | object (string) | Numerical |
| `signup_date` | object (string) | Date/Time |
| `quantity` | float64 | Numerical (integer count) |
| `notes` | float64 (all NaN) | String |
| `gender`, `city`, `membership_type`, `payment_method` | object | Categorical |

We'll fix each of these in turn.


## 2. Numerical Data

**Numerical data** represents quantities that support arithmetic (sums, means, comparisons). It splits into:
- **Continuous** (can take any value in a range) — e.g. `annual_income`, `purchase_amount`.
- **Discrete** (countable, whole numbers) — e.g. `quantity`, `rating`.

In this dataset, `annual_income` and `rating` already load as numeric (`float64`/`int64`). `age`, `purchase_amount`, and `quantity` need attention, which we handle below.


In [6]:
df[['annual_income', 'rating']].describe()

,annual_income,rating
count,9.430000e+02,1000.000000
mean,3.251967e+06,2.986000
std,5.633967e+07,1.474442
min,-5.000000e+04,-1.000000
25%,5.238569e+04,2.000000
50%,6.592116e+04,3.000000
75%,7.913065e+04,4.000000
max,1.000000e+09,10.000000


## 3. Categorical Data

**Categorical data** represents a value drawn from a fixed, usually small, set of possible labels rather than a measurable quantity. `gender`, `city`, `membership_type`, and `payment_method` are all categorical here — pandas currently stores them as generic `object` (string), which works but doesn't capture that they're a limited set of categories.


In [7]:
for col in ['gender', 'city', 'membership_type', 'payment_method']:
    print(f"{col}: {df[col].nunique(dropna=True)} unique values -> {df[col].dropna().unique()[:6]}")

gender: 7 unique values -> <ArrowStringArray>
['female', 'Female', 'Male', 'male', 'M', 'MALE']
Length: 6, dtype: str
city: 25 unique values -> <ArrowStringArray>
['Bengaluru ', 'Delhi ', ' Delhi', 'chennai', 'Chennai', 'delhi']
Length: 6, dtype: str
membership_type: 4 unique values -> <ArrowStringArray>
['Gold', 'Silver', 'Platinum', 'Bronze']
Length: 4, dtype: str
payment_method: 6 unique values -> <ArrowStringArray>
['Debit Card', 'Net Banking', 'Credit Card', 'UPI', 'COD', 'crypto']
Length: 6, dtype: str


Converting a column with a small, fixed set of values to pandas' dedicated `category` dtype (rather than leaving it as generic `object`) saves memory and lets pandas enforce/understand the fixed set of categories.


In [8]:
before_mem = df['membership_type'].memory_usage(deep=True)
df['membership_type'] = df['membership_type'].astype('category')
after_mem = df['membership_type'].memory_usage(deep=True)
print(f"Memory before: {before_mem} bytes")
print(f"Memory after:  {after_mem} bytes")
print(df['membership_type'].dtype)

Memory before: 13737 bytes
Memory after:  1189 bytes
category


## 4. Boolean Data

**Boolean data** holds exactly two states: `True`/`False` (often stored on disk as `Yes`/`No`, `1`/`0`, or similar). None of the raw columns here are boolean yet, but we can *derive* one — a natural example is flagging whether a customer is a "premium" member (Gold or Platinum).


In [9]:
df['is_premium_member'] = df['membership_type'].isin(['Gold', 'Platinum'])
print(df['is_premium_member'].dtype)
df[['membership_type', 'is_premium_member']].head()

bool


,membership_type,is_premium_member
0,Gold,True
1,Silver,False
2,Gold,True
3,Silver,False
4,Silver,False


This is a clean boolean column: exactly two possible values, stored as pandas' native `bool` dtype, ready to be used directly as a numeric 0/1 feature if needed (`True`/`False` behave as `1`/`0` in arithmetic).


## 5. Date/Time Data

**Date/Time data** represents points in time and needs a dedicated `datetime64` type to support date arithmetic (e.g. "days since signup"), sorting, and extraction of components (year, month, weekday). `signup_date` is currently plain text, and — as we saw in Notebook 1 — mixes multiple formats.


In [10]:
print("Sample raw signup_date values:")
print(df['signup_date'].dropna().unique()[:10])

Sample raw signup_date values:
<ArrowStringArray>
[ '08-22-2021', '16 Jun 2020', '28 Sep 2020', '10 Sep 2023',  '07/01/2021',
  '01-19-2021', '30 Jan 2023',  '07-21-2023',  '13/09/2019',  '07/10/2020']
Length: 10, dtype: str


We'll convert this properly in the "Converting Data Types" section below, since it needs `to_datetime()` with some care due to the mixed formats.


## 6. String Data

**String data** is free-form or semi-structured text — not meant to be treated as a fixed category or a number. In this dataset, `notes` is intended to hold free-text (though every value happens to be missing), and `customer_id`, while numeric-looking, functions as a string/label identifier rather than a quantity — it wouldn't make sense to average or sum customer IDs.


In [11]:
print("customer_id dtype:", df['customer_id'].dtype)
print("Does averaging customer_id make sense? Example (meaningless):", df['customer_id'].mean())

customer_id dtype: int64
Does averaging customer_id make sense? Example (meaningless): 100478.341


The fact that `.mean()` runs without error but produces a meaningless number is exactly why identifiers like `customer_id` should be explicitly treated as strings/labels, not numeric quantities, even though they happen to be stored as integers.


In [12]:
df['customer_id'] = df['customer_id'].astype(str)
print(df['customer_id'].dtype)
df['customer_id'].head()


str


0    100508
1    100819
2    100453
3    100369
4    100243
Name: customer_id, dtype: str

## 7. Converting Data Types

Pandas provides a few different tools for type conversion, and picking the right one matters:

- **`astype()`** — direct, "hard" type casting. Fast, but fails immediately (raises an error) if any value can't be converted.
- **`pd.to_numeric()`** — purpose-built for numeric conversion, with an `errors=` parameter to control what happens when a value can't be converted.
- **`pd.to_datetime()`** — purpose-built for date/time conversion, with format detection and an `errors=` parameter as well.

The next few sections demonstrate each, plus the requested conversion examples.


## 8. `astype()`

`astype()` performs a direct type cast. It's the right tool when you're confident every value is already convertible.

**Example: Integer → Float** — `quantity` is a naturally discrete/integer quantity (you can't buy 2.5 items), but it loaded as `float64` purely because of missing values (pandas can't store `NaN` in an integer column without special handling).


In [13]:
print("quantity dtype:", df['quantity'].dtype)
print("Missing values in quantity:", df['quantity'].isnull().sum())
df['quantity'].head()

quantity dtype: float64
Missing values in quantity: 20


0    1.0
1    1.0
2    2.0
3    9.0
4    1.0
Name: quantity, dtype: float64

If we tried to force this straight to a standard integer type while `NaN`s are still present, it would fail:


In [14]:
try:
    df['quantity'].astype(int)
except Exception as e:
    print(f"{type(e).__name__}: {e}")

IntCastingNaNError: Cannot convert non-finite values (NA or inf) to integer.Replace or remove non-finite values or cast to an integer typethat supports these values (e.g. 'Int64')


This is expected — standard integer types can't hold missing values. Pandas' **nullable integer type**, `Int64` (capital I), solves this by allowing `NaN` alongside whole numbers:


In [15]:
df['quantity'] = df['quantity'].astype('Int64')
print(df['quantity'].dtype)
df['quantity'].head()

Int64


0    1
1    1
2    2
3    9
4    1
Name: quantity, dtype: Int64

**Example: Numeric → Categorical** — `rating` is a small set of discrete numeric scores. We can bucket it into ordered categorical labels, which is often more interpretable for reporting and certain models than the raw number:


In [16]:
df['rating_level'] = pd.cut(
    df['rating'],
    bins=[-np.inf, 1, 3, np.inf],
    labels=['Low', 'Medium', 'High']
)
print(df['rating_level'].dtype)
df[['rating', 'rating_level']].head(8)

category


,rating,rating_level
0,1,Low
1,4,High
2,4,High
3,4,High
4,4,High
5,3,Medium
6,3,Medium
7,4,High


## 9. `pd.to_numeric()`

`to_numeric()` is purpose-built for converting to numbers and, unlike a raw `astype()`, gives control over what happens when a value *can't* be converted via the `errors` parameter (`'raise'`, `'coerce'`, or `'ignore'`).

**Example: String → Numeric (`age`)** — `age` loads as text purely because of how it was stored; the values themselves are numeric.


In [17]:
print("age dtype before:", df['age'].dtype)
df['age'] = pd.to_numeric(df['age'], errors='coerce')
print("age dtype after: ", df['age'].dtype)
df['age'].describe()

age dtype before: str
age dtype after:  float64


count    947.00000
mean      38.37698
std       14.55290
min       -5.00000
25%       30.00000
50%       38.00000
75%       45.00000
max      200.00000
Name: age, dtype: float64

**Example: String → Numeric (`purchase_amount`)** — this one is trickier: some values include a leading `$`, which `to_numeric()` can't parse directly.


In [18]:
print("Values that would fail a direct numeric conversion:")
print(df.loc[df['purchase_amount'].str.contains(r'\$', na=False), 'purchase_amount'].head())

Values that would fail a direct numeric conversion:
8       $41.88
21      $95.85
43      $23.85
96     $180.58
125      $52.5
Name: purchase_amount, dtype: str


In [19]:
df['purchase_amount'] = (
    df['purchase_amount']
    .str.replace('$', '', regex=False)
    .str.strip()
)
df['purchase_amount'] = pd.to_numeric(df['purchase_amount'], errors='coerce')
print("purchase_amount dtype:", df['purchase_amount'].dtype)
df['purchase_amount'].describe()

purchase_amount dtype: float64


count     1000.000000
mean       238.573880
std       1641.123073
min       -100.000000
25%         62.805000
50%        104.040000
75%        171.362500
max      25000.000000
Name: purchase_amount, dtype: float64

## 10. `pd.to_datetime()`

`to_datetime()` converts text into proper `datetime64` values, and can often auto-detect formats — including mixed formats within the same column, which is exactly the situation `signup_date` presents.

**Example: String → Date (`signup_date`)**


In [20]:
print("Raw formats present:")
print(df['signup_date'].dropna().unique()[:10])

Raw formats present:
<ArrowStringArray>
[ '08-22-2021', '16 Jun 2020', '28 Sep 2020', '10 Sep 2023',  '07/01/2021',
  '01-19-2021', '30 Jan 2023',  '07-21-2023',  '13/09/2019',  '07/10/2020']
Length: 10, dtype: str


In [21]:
df['signup_date'] = pd.to_datetime(df['signup_date'], format='mixed', errors='coerce')
print("signup_date dtype:", df['signup_date'].dtype)
df['signup_date'].head(10)

signup_date dtype: datetime64[us]


0   2021-08-22
1   2020-06-16
2   2020-09-28
3   2023-09-10
4   2021-07-01
5   2021-01-19
6   2023-01-30
7   2023-07-21
8   2019-09-13
9   2020-07-10
Name: signup_date, dtype: datetime64[us]

With `format='mixed'`, pandas infers the format row by row instead of assuming one fixed format for the whole column — exactly what's needed here since we confirmed multiple date formats coexist. Any value that still can't be parsed becomes `NaT` (pandas' "missing" marker for datetimes), rather than crashing the whole conversion.


In [22]:
print("Rows where signup_date could not be parsed:", df['signup_date'].isnull().sum())

Rows where signup_date could not be parsed: 16


Once converted, we get date arithmetic and component extraction for free — something impossible while the column was plain text:


In [23]:
print("Earliest signup:", df['signup_date'].min())
print("Latest signup:  ", df['signup_date'].max())
df['signup_year'] = df['signup_date'].dt.year
df[['signup_date', 'signup_year']].dropna().head()

Earliest signup: 2019-01-03 00:00:00
Latest signup:   2023-12-30 00:00:00


,signup_date,signup_year
0,2021-08-22,2021.0
1,2020-06-16,2020.0
2,2020-09-28,2020.0
3,2023-09-10,2023.0
4,2021-07-01,2021.0


## 11. Handling Invalid Conversions

Every conversion above used `errors='coerce'` deliberately. The `errors` parameter controls what happens when a value can't be converted:

- **`errors='raise'`** (default): stop immediately and throw an exception. Useful when you want to be alerted to any unexpected value.
- **`errors='coerce'`**: replace unconvertible values with `NaN`/`NaT` and continue. Useful when you expect some bad values and want to handle them as missing data afterward.
- **`errors='ignore'`**: leave unconvertible values (and the whole column) unchanged if *any* value fails. (Deprecated in recent pandas versions — prefer `coerce` plus explicit handling.)

The trade-off: `coerce` is convenient but can silently hide problems if you don't check afterward. Best practice is to always check how many values got coerced to `NaN`, so a conversion that failed doesn't get mistaken for a value that was legitimately missing.


In [24]:
print("age:              coerced to NaN ->", df['age'].isnull().sum())
print("purchase_amount:  coerced to NaN ->", df['purchase_amount'].isnull().sum())
print("signup_date:      coerced to NaT ->", df['signup_date'].isnull().sum())

age:              coerced to NaN -> 53
purchase_amount:  coerced to NaN -> 0
signup_date:      coerced to NaT -> 16


If `purchase_amount` or `age` had coerced values, that would mean some raw entries were fundamentally unparseable (e.g. text like `"unknown"` instead of a number) — worth investigating individually rather than assuming they're simple missing values, since they represent a different kind of problem (malformed input, not absent input).


## 12. Detecting Incorrect Data Types

Beyond just reading `df.dtypes`, a few practical checks help catch type problems that aren't obvious at a glance:


In [25]:
for col in df.select_dtypes(include='object').columns:
    converted = pd.to_numeric(df[col], errors='coerce')
    pct_numeric = converted.notnull().mean() * 100
    if pct_numeric > 50:
        print(f"'{col}' is stored as text but {pct_numeric:.0f}% of values look numeric -> likely mistyped")

'customer_id' is stored as text but 100% of values look numeric -> likely mistyped


C:\Users\hemak\AppData\Local\Temp\ipykernel_15264\2251686800.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include='object').columns:


In [26]:
for col in df.select_dtypes(include='object').columns:
    n_unique = df[col].nunique(dropna=True)
    print(f"{col}: {n_unique} unique values out of {len(df)} rows")

customer_id: 950 unique values out of 1000 rows
gender: 7 unique values out of 1000 rows
city: 25 unique values out of 1000 rows
payment_method: 6 unique values out of 1000 rows


C:\Users\hemak\AppData\Local\Temp\ipykernel_15264\2455884035.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include='object').columns:


In [27]:
df.dtypes

customer_id                     str
age                         float64
gender                          str
annual_income               float64
city                            str
membership_type            category
purchase_amount             float64
quantity                      Int64
signup_date          datetime64[us]
payment_method                  str
rating                        int64
notes                       float64
is_premium_member              bool
rating_level               category
signup_year                 float64
dtype: object

After these conversions:
- `age`, `purchase_amount` → proper numeric types.
- `quantity` → nullable integer (`Int64`).
- `signup_date` → proper `datetime64`.
- `membership_type` → `category`.
- `customer_id` → explicit string/label, not a numeric quantity.
- `is_premium_member` → boolean.
- `rating_level` → ordered categorical, derived from a numeric score.

`gender`, `city`, and `payment_method` are still `object` type here — they're correctly *categorical in nature*, but their raw labels still have the consistency issues (casing, whitespace) surfaced in Notebook 1. Converting them to `category` dtype is best done *after* that cleanup, so we don't lock in duplicate categories like `"Male"` and `"male"` as if they were genuinely different — that step belongs to the next preprocessing notebook.


## 13. Why Correct Data Types Are Important

- **Correctness of operations:** arithmetic, comparisons, and aggregations only behave correctly on the right type — averaging a numeric-looking string column either fails or (if pandas coincidentally succeeds) risks silently wrong results.
- **Enables date/time features:** operations like "days since signup" or "signup month" are only possible once a column is a true `datetime64`, not text.
- **Memory efficiency:** the `category` dtype for low-cardinality columns (like `membership_type`) uses substantially less memory than storing every value as a full string.
- **Model compatibility:** most ML algorithms require numeric input; encoding categorical variables and confirming numeric columns are truly numeric is a prerequisite, not an optional nicety.
- **Prevents silent bugs:** a column like `customer_id` stored as an integer can be accidentally summed or averaged without error, producing a nonsensical result that's easy to miss — explicit typing prevents this class of mistake.
- **Correct sorting and comparison:** dates stored as text sort lexicographically (`"2 Jan"` sorts before `"10 Jan"`), which is wrong; as real `datetime64` values they sort chronologically, as expected.

Getting data types right early prevents a long tail of subtle, hard-to-diagnose errors later in the pipeline — it's one of the highest-leverage, lowest-effort steps in preprocessing.
